# FormReader - TÜBİTAK 3501 Projesi

## Hücre Rehberi
- **Hücre 1-4**: Form işleme (opsiyonel)
- **Hücre 5**: Sentetik veri üretimi (50K) - ~30-40 dk
- **Hücre 6**: Model eğitimi (Deney #003 Bug Fix) - ~7-8 saat
- **Hücre 7**: Sonuç testi (DENEY değişkenini değiştirerek farklı deneyleri test et)

---
**GECE İÇİN**: Sadece Hücre 6'yı çalıştır! (veri zaten mevcut)

**VERSİYONLAMA**: Her deney ayrı klasöre kaydedilir (`deney002/`, `deney003/`, ...)

In [ ]:
# ============================================================
# HÜCRE 1: Kütüphaneleri Yükle
# ============================================================
import sys
import os
import cv2
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('../src')) 

from preprocessing import find_boxes
from utils import parse_xml_labels, match_and_crop
from ocr_engine import OCREngine

def show(img, title="Resim"):
    plt.figure(figsize=(10,10))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

print("[OK] Kütüphaneler yüklendi")

In [ ]:
# ============================================================
# HÜCRE 2: Formu Oku ve Kutuları Bul (OPSİYONEL)
# ============================================================
IMAGE_PATH = "../data/raw/S25C-925103023031.jpg"

processed_image, boxes = find_boxes(IMAGE_PATH)
print(f"Toplam {len(boxes)} adet kutu bulundu.")
show(processed_image, "Bulunan Kutular")

In [ ]:
# ============================================================
# HÜCRE 3: Etiketleri Eşleştir ve Kes (OPSİYONEL)
# ============================================================
XML_PATH = "../data/xml_labels/S25C-925103023031.xml"

xml_labels = parse_xml_labels(XML_PATH)
labeled_data = match_and_crop(processed_image, boxes, xml_labels, output_folder="../data/processed")

for item in labeled_data[:5]:
    print(f"Etiket: {item['label']}")
    show(item['roi_image'], item['label'])

In [ ]:
# ============================================================
# HÜCRE 4: Base Model ile Test (OPSİYONEL)
# ============================================================
ocr = OCREngine()

print("\n--- BASE MODEL OKUMA SONUÇLARI ---")
for item in labeled_data[:10]:
    text = ocr.predict(item['roi_image'])
    print(f"Kutu: {item['label']} -> Okunan: {text}")

In [ ]:
# ============================================================
# HÜCRE 5: SENTETİK VERİ ÜRETİMİ (200.000 ADET)
# ============================================================
# Tahmini süre: ~2 saat
# Disk: ~8-10 GB
# ============================================================

from data_generator import generate_dataset
import pandas as pd

FONT_DIR = "../data/fonts"
OUTPUT_DIR = "../data/synthetic"
COUNT = 200000  # Deney #005 için 200K

print("="*60)
print("SENTETİK VERİ ÜRETİMİ BAŞLIYOR")
print(f"Hedef: {COUNT:,} görüntü")
print("="*60)

csv_path = generate_dataset(FONT_DIR, OUTPUT_DIR, count=COUNT)

# Kontrol
df = pd.read_csv(csv_path)
print(f"\n[TAMAMLANDI] Toplam: {len(df):,} veri")
print(f"\nKategori dağılımı:")
print(df['category'].value_counts())

In [1]:
# ============================================================
# HÜCRE 6: MODEL EĞİTİMİ (Deney #006 - PaddleOCR)
# ============================================================
# Mimari: PP-OCRv4 Server Recognition (SVTR + CTC)
# Decoder: CTC (karakter bazlı - tokenizer sorunu YOK!)
# Türkçe dict.txt ile ş,ğ,ü,ı,ö,ç tek sınıf olarak tanımlı
# GPU: RTX 4070 Super 12GB
# ============================================================

import importlib
import sys
sys.path.insert(0, '../src')

import paddle_trainer
importlib.reload(paddle_trainer)
from paddle_trainer import train

EPOCHS = 10
BATCH_SIZE = 32

print("="*60)
print("DENEY #006: PaddleOCR Fine-Tuning")
print("Mimari: PP-OCRv4 Server (SVTR + CTC)")
print(f"Epochs: {EPOCHS} | Batch: {BATCH_SIZE}")
print("="*60)

train(epochs=EPOCHS, batch_size=BATCH_SIZE)

DENEY #006: PaddleOCR Fine-Tuning
Mimari: PP-OCRv4 Server (SVTR + CTC)
Epochs: 10 | Batch: 32
Deney #006: PaddleOCR Fine-Tuning
Mimari: PP-OCRv4 Server Recognition (SVTR)
Decoder: CTC (karakter bazlı)


C:\Users\User\anaconda3\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


[OK] PaddlePaddle 3.0.0
     GPU: True
     CUDA: 1 GPU bulundu
[OK] PaddleOCR repo: C:\Users\User\Desktop\FormReader\PaddleOCR
[OK] Pretrained model mevcut: C:\Users\User\Desktop\FormReader\PaddleOCR\pretrained_models\ch_PP-OCRv4_rec_server_train
[OK] Veri: Train=160,000 | Val=40,000
[OK] Dict: C:\Users\User\Desktop\FormReader\data\turkish_dict.txt

------------------------------------------------------------
Eğitim başlıyor...
Config: configs/rec/PP-OCRv4/PP-OCRv4_server_rec.yml
Pretrained: C:\Users\User\Desktop\FormReader\PaddleOCR\pretrained_models\ch_PP-OCRv4_rec_server_train
Epochs: 10
Batch size: 32
Çıktı: C:\Users\User\Desktop\FormReader\models\paddle-turkish-handwritten\deney006
------------------------------------------------------------
INFO: Could not find files for the given pattern(s).
C:\Users\User\anaconda3\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required.

In [1]:
# ============================================================
# HÜCRE 7: SONUÇ TESTİ (PaddleOCR - tools/infer_rec.py)
# ============================================================

import subprocess, sys, os, csv, random, unicodedata

PROJECT_DIR = os.path.abspath('..')
DENEY = "deney006"
model_dir = os.path.join(PROJECT_DIR, "models", "paddle-turkish-handwritten", DENEY)
config_path = os.path.join(PROJECT_DIR, "configs", "turkish_rec.yml")
data_dir = os.path.join(PROJECT_DIR, "data", "synthetic")
csv_path = os.path.join(data_dir, "metadata.csv")
paddleocr_dir = os.path.join(PROJECT_DIR, "PaddleOCR")

print(f"Test edilen deney: {DENEY}")
print(f"Model: {model_dir}")

# CSV'den test örnekleri seç
categories = ['number', 'turkish_text', 'date', 'mixed', 'code']
random.seed(42)
test_items = []
with open(csv_path, 'r', encoding='utf-8') as f:
    rows_by_cat = {}
    for row in csv.DictReader(f):
        cat = row['category']
        if cat in categories:
            rows_by_cat.setdefault(cat, []).append(row)
for cat in categories:
    if cat in rows_by_cat:
        for row in random.sample(rows_by_cat[cat], min(5, len(rows_by_cat[cat]))):
            test_items.append({
                'category': cat,
                'file_name': row['file_name'],
                'text': unicodedata.normalize('NFC', str(row['text']).strip())
            })

# UTF-8 encoding için environment
env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"

print(f"\n{'='*60}")
print(f"SENTETİK VERİ TESTİ (PaddleOCR - Deney #006)")
print(f"{'='*60}")

correct = 0
total = 0
current_cat = ""

for item in test_items:
    cat = item['category']
    if cat != current_cat:
        print(f"\n--- {cat.upper()} ---")
        current_cat = cat
    
    img_path = os.path.join(data_dir, item['file_name'])
    expected = item['text']
    
    cmd = [
        sys.executable,
        os.path.join(paddleocr_dir, "tools", "infer_rec.py"),
        "-c", config_path,
        "-o", f"Global.infer_img={img_path}",
    ]
    
    result = subprocess.run(
        cmd, capture_output=True, text=True,
        encoding='utf-8', errors='replace',
        cwd=PROJECT_DIR, env=env
    )
    
    # Parse: "result: TEXT\tSCORE"
    predicted = ""
    confidence = 0.0
    for line in result.stdout.split('\n'):
        if 'result:' in line:
            try:
                parts = line.split('result:')[1].strip().split('\t')
                predicted = parts[0].strip()
                if len(parts) > 1:
                    confidence = float(parts[1].strip())
            except:
                pass
    
    match = "✓" if predicted == expected else "✗"
    if match == "✓":
        correct += 1
    total += 1
    
    print(f"{match} Beklenen: {expected:<25} | Tahmin: {predicted:<25} | Güven: {confidence:.4f}")

print(f"\n{'='*60}")
print(f"DENEY: {DENEY}")
print(f"DOĞRULUK: {correct}/{total} = {100*correct/total:.1f}%")
print(f"{'='*60}")

Test edilen deney: deney006
Model: C:\Users\User\Desktop\FormReader\models\paddle-turkish-handwritten\deney006

SENTETİK VERİ TESTİ (PaddleOCR - Deney #006)

--- NUMBER ---
✓ Beklenen: 666                       | Tahmin: 666                       | Güven: 0.8130
✓ Beklenen: 593                       | Tahmin: 593                       | Güven: 1.0000
✓ Beklenen: %46                       | Tahmin: %46                       | Güven: 0.9958
✓ Beklenen: 30.3                      | Tahmin: 30.3                      | Güven: 0.9979
✓ Beklenen: 600                       | Tahmin: 600                       | Güven: 1.0000

--- TURKISH_TEXT ---
✓ Beklenen: 2 LT                      | Tahmin: 2 LT                      | Güven: 1.0000
✓ Beklenen: Kontaktör arızası         | Tahmin: Kontaktör arızası         | Güven: 0.9680
✓ Beklenen: Çağla KURT                | Tahmin: Çağla KURT                | Güven: 0.9794
✓ Beklenen: Hat 2 durdu               | Tahmin: Hat 2 durdu               | Güven: 0.

In [1]:
# ============================================================
# HÜCRE 8: GERÇEK FORM TESTİ (PaddleOCR)
# ============================================================
# data/processed/ klasöründeki gerçek form kutularını oku
# Bu kutular data/raw/S25C-925103023031.jpg formundan kesilmiş
# ============================================================

import subprocess, sys, os, glob

PROJECT_DIR = os.path.abspath('..')
config_path = os.path.join(PROJECT_DIR, "configs", "turkish_rec.yml")
paddleocr_dir = os.path.join(PROJECT_DIR, "PaddleOCR")
processed_dir = os.path.join(PROJECT_DIR, "data", "processed")

env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"

# Her kategoriden birkaç resim test et
test_categories = [
    "input_durus_dakika",       # Sayısal: duruş dakikaları
    "input_durus_aciklamalar",  # El yazısı: duruş açıklamaları
    "input_hat_sorumlusu",      # El yazısı: operatör isimleri
    "input_durus_suresi",       # Sayısal: duruş süreleri
]

print("=" * 70)
print("GERÇEK FORM TESTİ (PaddleOCR - data/raw/ formundan kesilmiş kutular)")
print("=" * 70)

for cat in test_categories:
    cat_dir = os.path.join(processed_dir, cat)
    if not os.path.exists(cat_dir):
        print(f"\n[!] Klasör bulunamadı: {cat}")
        continue
    
    images = sorted(glob.glob(os.path.join(cat_dir, "*.jpg")))[:5]  # İlk 5
    if not images:
        continue
    
    print(f"\n--- {cat.upper()} ({len(images)} örnek) ---")
    
    for img_path in images:
        cmd = [
            sys.executable,
            os.path.join(paddleocr_dir, "tools", "infer_rec.py"),
            "-c", config_path,
            "-o", f"Global.infer_img={img_path}",
        ]
        
        result = subprocess.run(
            cmd, capture_output=True, text=True,
            encoding='utf-8', errors='replace',
            cwd=PROJECT_DIR, env=env
        )
        
        predicted = ""
        confidence = 0.0
        for line in result.stdout.split('\n'):
            if 'result:' in line:
                try:
                    parts = line.split('result:')[1].strip().split('\t')
                    predicted = parts[0].strip()
                    if len(parts) > 1:
                        confidence = float(parts[1].strip())
                except:
                    pass
        
        img_name = os.path.basename(img_path)
        print(f"  {img_name:<40} → {predicted:<30} (güven: {confidence:.4f})")

print(f"\n{'=' * 70}")
print("NOT: Bu gerçek el yazısı formlardan kesilmiş kutulardır.")
print("Beklenen değerler bilinmiyor - sonuçları görsel olarak doğrulayın.")
print(f"{'=' * 70}")

GERÇEK FORM TESTİ (PaddleOCR - data/raw/ formundan kesilmiş kutular)

--- INPUT_DURUS_DAKIKA (5 örnek) ---
  input_durus_dakika_101.jpg               → 2                              (güven: 0.7704)
  input_durus_dakika_109.jpg               → 40                             (güven: 0.7063)
  input_durus_dakika_112.jpg               → %6                             (güven: 0.6640)
  input_durus_dakika_117.jpg               → 0                              (güven: 0.5843)
  input_durus_dakika_122.jpg               → 2                              (güven: 0.6837)

--- INPUT_DURUS_ACIKLAMALAR (5 örnek) ---
  input_durus_aciklamalar_0.jpg            → 3                              (güven: 0.9227)
  input_durus_aciklamalar_102.jpg          → 0.0                            (güven: 0.0000)
  input_durus_aciklamalar_105.jpg          → Çerk rmem 0eer                 (güven: 0.5976)
  input_durus_aciklamalar_110.jpg          → 28p                            (güven: 0.5982)
  input_durus_aciklama

In [1]:
# ============================================================
# HÜCRE 9: GERÇEK FORM FULL PIPELINE (Detection + Recognition)
# ============================================================
# 1. Paddle Inference API ile metin kutularını bul (DLL çakışması yok!)
# 2. Her kutuyu kes, infer_rec.py ile oku (subprocess, çalışıyor)
# 3. Sonuçları koordinat + metin olarak göster
# ============================================================

import importlib
import sys
sys.path.insert(0, '../src')

import paddle_pipeline
importlib.reload(paddle_pipeline)
from paddle_pipeline import process_form

import glob, os

raw_dir = os.path.join('..', 'data', 'raw')
forms = sorted(glob.glob(os.path.join(raw_dir, '*.jpg')))

# İlk 1 form ile test (her kutu için subprocess çalışacak, biraz yavaş)
MAX_FORMS = 1

print(f"Toplam form: {len(forms)}")
print(f"Test edilecek: {MAX_FORMS}")
print(f"{'='*70}")

for form_path in forms[:MAX_FORMS]:
    results = process_form(form_path)
    print(f"\n  TOPLAM: {len(results)} metin okundu")

print(f"\n{'='*70}")
print("Sonuçları görsel olarak doğrulayın.")

Toplam form: 20
Test edilecek: 1

[Pipeline] S25C-925103023000.jpg


C:\Users\User\anaconda3\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:711: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


  [Detection] 93 metin kutusu bulundu (3.0s)
  [Recognition] 50 metin okundu (261.0s)
  ( 528, 104) 0.0                                 güven:0.00
  (1770, 111) 1 B1 LB                             güven:0.54
  (1971, 111) 22                                  güven:0.56
  ( 126, 174) ela3                                güven:0.31
  (1021, 221) YZ kaçğlı                           güven:0.64
  ( 462, 225) 8                                   güven:0.65
  (1770, 236) İ6 ERĞ6                             güven:0.59
  (1964, 236) 8la                                 güven:0.37
  ( 400, 360) EIM                                 güven:0.54
  ( 553, 360) İBEEĞEU                             güven:0.71
  (1412, 363) 5EEZ                                güven:0.57
  ( 148, 371) BÜN                                 güven:0.69
  ( 725, 378) KAPBK EŞELIR                        güven:0.82
  ( 900, 378) ETKET ŞELIR                         güven:0.90
  (1072, 378) SHRNK ÇILAR                         güven:0.77